In [11]:
# Criar um navegador
# Importar/visualizar a base de dados
# Para cada item dentro da nossa base de dados, para cada produto
#     Procurar esse produto no Google Shooping
#         Verificar se algum dos produtos do Google Shopping está dentro da minha faixa de preço
#     Procurar esse produto em alguma outra plataforma que eu desejar
#         Verificar se algum dos produtos dessa outra plataforma está dentro da minha faixa de preço
# Salvar as ofertas boas em um dataframe (tabela)
# Exportar para o Excel
# Enviar por e-mail o resultado da tabela com os melhores preços encontrados

In [12]:
# Bibliotecas

import time
import os
import random
from time import sleep

from pandas.core.dtypes.missing import na_value_for_dtype
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from fake_useragent import UserAgent
import pandas as pd

In [13]:
# Configuração portátil de caminhos (Sem expor dados pessoais)

caminho_projeto = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
pasta_perfil_robo = os.path.join(caminho_projeto, "Perfil_Do_Robo")

options = Options()
options.add_argument(f"--user-data-dir={pasta_perfil_robo}")
options.add_argument("--profile-directory=Default")

# Criar um navegador

ua = UserAgent()
user_agent_aleatorio = ua.random
options.add_argument(f"user-agent={user_agent_aleatorio}")

# Remove os rastros de automação padrão do Chrome
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

navegador = webdriver.Chrome(options=options)

# Importar base de dados

tabela_prod = pd.read_excel('Produtos\\Tabela_Produtos.xlsx')
display(tabela_prod)

,Nome,Termos Banidos,Preço Min,Preço Max
0,RX 9070XT,9060XT 9060 5070,3800,4500
1,Himalayan 450,Himalayan 411,29000,35000


In [14]:
def busca_google_shopping(navegador, produto, termos_banidos, preco_minimo, preco_maximo):
    produto = produto.lower()
    termos_banidos = termos_banidos.lower()
    lista_termos_banidos = termos_banidos.split(' ')
    lista_termos_produto = produto.split(' ')

    lista_resultados = []

    # Entrar no navegador/ Google/ Pesquisar produto/ Entrar na aba shopping/ Coletar informações

    navegador.get('https://www.google.com/')
    sleep(2)
    campo_busca = navegador.find_element(By.NAME, 'q')

    # Envia o texto para o campo

    for letra in produto:
        campo_busca.send_keys(letra)
        sleep(random.uniform(0.1, 0.3)) # Pausa aleatória entre as letras

    sleep(0.5)
    campo_busca.send_keys(Keys.ENTER)

    # Entrar na aba shopping

    sleep(3)
    elementos = navegador.find_elements('class name', 'mXwfNd')

    for item in elementos:
        if 'Shopping' in item.text:
            item.click()
            break

    # Pegar as informações dos produtos

    lista_buscas = navegador.find_elements('class name', 'wOPJ9c')

    for resultado in lista_buscas:
        nome = resultado.find_element('class name', 'gkQHve').text
        nome = nome.lower()

        # Analisar se ele não tem nenhum termo banido

        tem_termo_banido = False
        for palavra in lista_termos_banidos:
            if palavra in nome:
                tem_termo_banido = True

        # Analisar se ele tem TODOS os termos do nome do produto

        tem_todos_termos_produto = True
        for palavra in lista_termos_produto:
            if palavra not in nome:
                tem_todos_termos_produto = False

        # Selecionar só os elementos que tem_termo_banido = False e ao mesmo tempo tem_todos_termos_produto = True

        if not tem_termo_banido and tem_todos_termos_produto:
            preco = resultado.find_element('class name', 'lmQWe').text
            preco = preco.replace('agora', '').replace('R$', '').replace(' ', '').replace('.', '').replace(',', '.')
            preco = float(preco)

            preco_minimo = float(preco_minimo)
            preco_maximo = float(preco_maximo)

        # Se o preço está entre o preco_minimo e preco_maximo

            if preco_minimo <= preco <= preco_maximo:
                lista_resultados.append((nome, preco))
    return lista_resultados

In [15]:
produto = 'RX 9070 XT'
termos_banidos = '9060XT 9060 5070'
preco_minimo = 3800
preco_maximo = 4500

lista_ofertas_google_shopping = busca_google_shopping(navegador, produto, termos_banidos, preco_minimo, preco_maximo)

print(lista_ofertas_google_shopping)

[('placa de vídeo xfx rx 9070 xt swift amd radeon', 4459.99), ('placa de vídeo asrock amd radeon rx 9070 xt challenger 16gb gddr6', 3970.03), ('placa de video xfx radeon rx 9070 xt swift white triple fan gaming edition', 4459.99), ('placa de vídeo powercolor reaper amd radeon rx 9070 xt 16gb gddr6', 4459.99)]
